In [1]:
!pip install -q -U transformers peft trl datasets bitsandbytes mlflow accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 94.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 97.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.8 MB/s eta 0:00:00
   ━━━━━

In [2]:
import os
import torch
import mlflow
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

os.environ["MLFLOW_EXPERIMENT_NAME"] = "quant_singularity"
mlflow.set_tracking_uri("file:./mlruns")

dataset_path = "/kaggle/input/datasets/aryanamittiwari/tattuuu/finetune_clean_cot.jsonl" 
dataset = load_dataset("json", data_files=dataset_path, split="train")

def format_instruction(example):
    return f"<|system|>\n{example['instruction']}</s>\n<|user|>\n{example['input']}</s>\n<|assistant|>\n{example['output']}</s>"

Generating train split: 0 examples [00:00, ? examples/s]

In [3]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading {model_id}")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

# Rank 8 provides enough expressivity to learn JSON syntax and CoT logic without overfitting to noise.
lora_config = LoraConfig(
    r=8, 
    lora_alpha=16, 
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [4]:
training_args = SFTConfig(
    output_dir="/kaggle/working/quant_slm_adapter",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=150, 
    warmup_steps=5,     
    lr_scheduler_type="cosine",
    report_to="mlflow", 
)

trainer = SFTTrainer(
    model=model, 
    train_dataset=dataset,
    formatting_func=format_instruction,
    processing_class=tokenizer,  
    args=training_args,
)

with mlflow.start_run(run_name="TinyLlama_LoRA_R8_CoT"):
    mlflow.log_param("confluence_cot_enabled", True)
    mlflow.log_param("dropped_poisoned_rows", 2)
    
    trainer.train()
    
    trainer.model.save_pretrained("/kaggle/working/quant_slm_adapter")
    tokenizer.save_pretrained("/kaggle/working/quant_slm_adapter")
    print("Training complete")

Applying formatting function to train dataset:   0%|          | 0/298 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/298 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/298 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,2.126821
20,1.367468
30,0.650178
40,0.479792
50,0.458421
60,0.446332
70,0.441202
80,0.439496
90,0.434487
100,0.431719


Training complete


In [5]:
import shutil
import os

shutil.make_archive('/kaggle/working/quant_slm_adapter', 'zip', '/kaggle/working/quant_slm_adapter')
shutil.make_archive('/kaggle/working/mlruns', 'zip', '/kaggle/working/mlruns')

print("Files in working directory after zipping:", os.listdir('/kaggle/working'))

Files in working directory after zipping: ['quant_slm_adapter', 'mlruns.zip', '.virtual_documents', 'quant_slm_adapter.zip', 'mlruns']
